# Notebook 49 - Unseen reconnaissance on CIC-IoMT-2024 with six more seeds

**Why this notebook.** Notebook 47 held out DoS, reconnaissance and spoofing on CIC-IoMT-2024 with three seeds. No cell met the pre-registered gap rule, but reconnaissance pointed the other way from CICIoT2023: the Fisher students raised it less than the dense reference on every seed (9% and 19% against 23% and 52%) and the magnitude students on two of three, with intervals too wide to read (Fisher +13.5 [-4.1, +31.0] and +32.0 [-5.1, +69.2] points). On CICIoT2023 the pruned students raised unseen reconnaissance more. Three seeds cannot say which way IoMT goes; this notebook runs six more.

**What it does.** GPU, about one and a half hours. Notebook 47's protocol unchanged, reconnaissance held out only, seeds 401, 503, 601, 701, 809, 907: per (network, seed) a teacher trained from scratch without the family (accepted by notebook 47's fixed rule), a dense reference with two further epochs, magnitude and Fisher students at 90% recovered for two epochs, every model recalibrated. The analysis pools them with notebook 47's three reconnaissance seeds.

**Readings, fixed in stage 2 before any result.** At nine seeds, per network and rule: notebook 47's gap condition (student raises the family at least 5 points less often, interval above zero, matched in distribution) and its mirror (student raises it more, interval below zero).

In [ ]:
# Stage 1 - bootstrap (GPU runtime)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, gc, time, copy, itertools, subprocess, importlib
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt

REPO = Path("/content/drive/MyDrive/SABER_Research/saber-ids")
IOMT_REPO = Path("/content/drive/MyDrive/IOMT_Compression_Research/iomt-compression-research")
os.chdir(REPO)
for p in (str(REPO), str(IOMT_REPO / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()
from src.saber.bridge_iomt import build_bridge, iomt_taxonomy
from src.saber.taxonomy import DEFAULT_COST_PROFILES
from src.saber.metrics import full_model_audit
from src.saber.surgery import prune_cnn1d_channels, profile_forward_flops, count_parameters, enumerate_cnn1d_channel_groups
from src.saber.leverage import magnitude_scores, gradient_saliency_scores
from src.saber.postg5 import calibrate_prefix_count

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"; IN32 = R / "32_iomt_bridge"
IN47 = R / "47_iomt_unseen_family"; OUT = R / "49_iomt_unseen_recon_seeds"; OUT.mkdir(parents=True, exist_ok=True)
DATA_CACHE = REPO / "data/iomt_bridge"; TEACHER_DIR = REPO / "models/iomt/lofo"; TEACHER_DIR.mkdir(parents=True, exist_ok=True)   # git-ignored
assert (IN32 / "bridge_manifest.json").exists() and (IN32 / "teacher_audit.json").exists(), "NB32 results missing; pull the repo"
assert (IN47 / "N47_runs.csv").exists() and (IN47 / "N47_PREREGISTRATION.json").exists(), "NB47 results missing; pull the repo"
print("device:", DEVICE, "| torch", torch.__version__)

In [ ]:
# Stage 2 - pre-registration (frozen: rerunning this cell can never change it after its commit)
PREREG = {
    "notebook": "49_iomt_unseen_recon_seeds",
    "question": ("on CIC-IoMT-2024, with nine seeds instead of three, does a pruned and recovered detector raise unseen reconnaissance less "
                 "often, more often, or as often as the dense detector trained and recovered the same way, at matched in-distribution risk"),
    "motivation": ("notebook 47 (three seeds): Fisher students raised unseen reconnaissance less on every seed (shallow 9.2% vs 22.7%, deep "
                   "19.5% vs 51.5%), magnitude students less on two seeds of three, no interval excluding zero; on CICIoT2023 (notebook 41, "
                   "five seeds) the magnitude students raised it more on both networks"),
    "design": {
        "protocol": "notebook 47 unchanged: same bridge, Setting construction, teacher training, acceptance rule, dense reference, scoring, prefix calibration, recovery, recalibration and audit",
        "held_out_families": ["recon"],
        "seeds": [401, 503, 601, 701, 809, 907],
        "pooling": "notebook 47's reconnaissance rows (seeds 101, 211, 307) plus this notebook's rows; paired by seed; nine seeds per network and rule",
        "acceptance_rule": "notebook 47's: benign escalation <= 0.15, attack recall >= 0.85, fine macro-F1 >= 0.40 on the remaining classes after recalibration; failing cells excluded and listed",
    },
    "readings": {
        "less": ("per network and rule: the paired gap (dense minus student, share raised as attack) is at least 0.05 with its 95% interval "
                 "above zero and the cell is matched (mean paired in-distribution HSR difference within 0.02): the student raises the family less"),
        "more": "per network and rule: the paired gap is negative with its 95% interval below zero: the student raises the family more",
        "level": "neither: the interval contains zero or the mean is below 0.05",
        "per_network": "pruned_raise_less if both rules read less; pruned_raise_more if both read more; mixed otherwise",
    },
    "expectation": {"Z1": "the Fisher student reads 'less' on at least one network (the three-seed direction holds)"},
    "reporting_rule": "every cell reported as measured; whichever way it reads, Section V of the paper states it",
    "budgets": [0.90], "methods": ["magnitude", "fisher"], "gap_threshold": 0.05, "match_threshold": 0.02,
    "teacher_epochs": 4, "recovery_epochs": 2, "no_test_access": True,
}
pre_path = OUT / "N49_PREREGISTRATION.json"
if pre_path.exists():
    print("pre-registration already exists; left untouched:", pre_path)
else:
    pre_path.write_text(json.dumps(PREREG, indent=2)); print("pre-registration written:", pre_path)
print(json.dumps(PREREG, indent=2))

ARCHS = ["shallow", "deep"]; FAMILIES = PREREG["design"]["held_out_families"]; SEEDS = PREREG["design"]["seeds"]
METHODS = PREREG["methods"]; BUDGETS = PREREG["budgets"]; PRIMARY = 0.90
GAP = PREREG["gap_threshold"]; MATCH = PREREG["match_threshold"]; T_EPOCHS = PREREG["teacher_epochs"]; R_EPOCHS = PREREG["recovery_epochs"]
TOL = 0.015; CAL_SEED = 2026; FISHER_BATCHES = 40; MIN_W = 8
ACCEPT = {"b2a_max": 0.15, "attack_recall_min": 0.85, "fine_f1_min": 0.40}
ALPHA = {"shallow": 0.5, "deep": 0.35}
NAME = {"magnitude": "magnitude", "fisher": "Fisher", "dense": "dense"}
FAMN = {"benign": "benign", "ddos": "DDoS", "dos": "DoS", "mqtt": "MQTT", "recon": "recon", "spoofing": "spoofing"}

In [ ]:
# Stage 2b - commit the pre-registration before any result exists (skipped automatically once results exist)
def _prereg_committed():
    rel = "results/saber/49_iomt_unseen_recon_seeds/N49_PREREGISTRATION.json"
    h = subprocess.run(["git", "log", "-1", "--format=%h", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    dirty = subprocess.run(["git", "status", "--porcelain", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    return bool(h) and not dirty


_results_exist = any(p.is_file() and p.name != "N49_PREREGISTRATION.json" for p in OUT.rglob("*"))
if _results_exist:
    assert _prereg_committed(), ("results exist but the pre-registration is not committed, so it can no longer be committed before any "
                                 "result; stop here and report it")
    print("results already exist in", OUT.name, "and the pre-registration is committed; nothing to do here")
else:
    os.chdir(REPO)
    r = subprocess.run(["python", "tools/commit_cell.py",
                        "NB49 pre-registration, committed before any result: unseen reconnaissance on CIC-IoMT-2024 with six more seeds (401, "
                        "503, 601, 701, 809, 907), the NB47 protocol unchanged (teachers from scratch without the family, fixed acceptance "
                        "rule, dense reference +2 epochs, magnitude and Fisher students at 90% recovered 2 epochs, all recalibrated), pooled "
                        "with NB47's three seeds to nine. Readings per network and rule: 'less' (gap >= 5 points, CI above zero, matched), "
                        "'more' (CI below zero), 'level'. Expectation Z1: Fisher reads 'less' on at least one network. No test access."],
                       capture_output=True, text=True)
    print(r.stdout); print(r.stderr)

In [ ]:
# Stage 3 - bridge and helpers (settings per held-out family)
TRAIN_LOADER, VAL_LOADER, CLASS_NAMES, FULL_TAX, MANIFEST = build_bridge(DATA_CACHE)
N_CLASSES = len(CLASS_NAMES); FAM_ALL = np.array(FULL_TAX.family_by_class); FAM_ORIG = list(MANIFEST["family_by_class"])
assert set(FAMILIES) <= set(FAM_ALL.tolist()), f"unknown family in {FAMILIES}; families are {sorted(set(FAM_ALL.tolist()))}"
Xt, Yt = TRAIN_LOADER.dataset.tensors; yt = Yt.numpy(); N_FEAT = int(Xt.shape[1])
Xv, Yv = VAL_LOADER.dataset.tensors; VAL_Y_ALL = Yv.numpy()
_rng = np.random.default_rng(0)
_idx = np.concatenate([_rng.permutation(np.where(VAL_Y_ALL == c)[0])[:4000] for c in range(N_CLASSES) if (VAL_Y_ALL == c).sum() > 0])
_idx = np.random.default_rng(12345).permutation(_idx)
EX_X_ALL = Xv[_idx]; EX_Y_ALL = VAL_Y_ALL[_idx]
del Xv, Yv
print("train rows", len(yt), "| eval rows", len(_idx), "| features", N_FEAT, "| families", sorted(set(FAM_ALL.tolist())))


class CNN1D(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(1, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(64),
                                  nn.Conv1d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(128))
        self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(128, n_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


class DeepCNN1D(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        def blk(i, o):
            return [nn.Conv1d(i, o, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(o)]
        self.conv = nn.Sequential(*blk(1, 64), *blk(64, 128), nn.MaxPool1d(2), *blk(128, 128), *blk(128, 256))
        self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(256, n_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


class Setting:
    '''Everything that depends on which family is held out: rows, labels, taxonomy, weights, calibration slice, evaluation rows.'''
    def __init__(self, family):
        self.family = family
        self.seen = [c for c in range(N_CLASSES) if FAM_ALL[c] != family]; self.unseen = [c for c in range(N_CLASSES) if FAM_ALL[c] == family]
        assert self.unseen, f"family {family} has no classes"
        self.remap = {c: i for i, c in enumerate(self.seen)}; self.names = [CLASS_NAMES[c] for c in self.seen]; self.K = len(self.seen)
        self.tax = iomt_taxonomy(self.names, [FAM_ORIG[c] for c in self.seen]); self.fam = np.array(self.tax.family_by_class); self.benign = self.names.index(self.tax.benign_class)
        self.seen_families = list(self.tax.families)
        keep = np.isin(yt, self.seen); self.train_idx = np.where(keep)[0]
        self.y_train = np.array([self.remap[c] for c in yt[keep]], dtype=np.int64); self.n_train = len(self.train_idx)
        self.ds = torch.utils.data.TensorDataset(Xt[torch.from_numpy(self.train_idx)], torch.from_numpy(self.y_train))
        counts = np.bincount(self.y_train, minlength=self.K); self.counts = counts
        self.class_w = {}
        for arch, alpha in ALPHA.items():
            w = np.zeros(self.K); w[counts > 0] = 1.0 / np.power(counts[counts > 0], alpha); w[counts > 0] /= w[counts > 0].mean()
            self.class_w[arch] = torch.tensor(w, dtype=torch.float32, device=DEVICE)
        cal = torch.randperm(self.n_train, generator=torch.Generator().manual_seed(CAL_SEED))[:50 * 1024]
        self.cal_loader = torch.utils.data.DataLoader(torch.utils.data.Subset(self.ds, cal.tolist()), batch_size=1024, shuffle=False)
        m_seen = np.isin(EX_Y_ALL, self.seen); m_un = np.isin(EX_Y_ALL, self.unseen)
        self.ex_x = EX_X_ALL[torch.from_numpy(m_seen)].to(DEVICE); self.ex_y = np.array([self.remap[c] for c in EX_Y_ALL[m_seen]], dtype=np.int64)
        self.un_x = EX_X_ALL[torch.from_numpy(m_un)].to(DEVICE); self.un_y = EX_Y_ALL[m_un]
        self.example = self.ex_x[:8].float(); self.fam_y = self.fam[self.ex_y]

    def loader(self, seed):
        return torch.utils.data.DataLoader(self.ds, batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(seed))

    def make(self, arch):
        return (CNN1D if arch == "shallow" else DeepCNN1D)(self.K)


def forward_logits(model, X):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X[i:i + 8192]).cpu() for i in range(0, len(X), 8192)]).numpy()


def train_epochs(model, loader, epochs, weights):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss(weight=weights)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); lossf(model(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
    return model.eval()


def recalibrate_bn(model, loader, n_batches=50):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.reset_running_stats(); m.momentum = None
    model.train()
    with torch.no_grad():
        for i, (xb, _) in enumerate(loader):
            if i >= n_batches: break
            model(xb.to(DEVICE))
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.momentum = 0.1
    model.eval(); return model


def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True); e = np.exp(z); return e / e.sum(axis=1, keepdims=True)


def audit(model, S):
    '''In-distribution audit on the remaining classes, plus the unseen-family audit.'''
    lg = forward_logits(model, S.ex_x); a = full_model_audit(lg, S.ex_y, S.tax, DEFAULT_COST_PROFILES); pf = S.fam[lg.argmax(axis=1)]
    out = {"seen_hsr_balanced": float(a["hsr_balanced_soc"]), "seen_hsr_miss": float(a["hsr_miss_sensitive"]), "seen_hsr_fatigue": float(a["hsr_alert_fatigue"]),
           "seen_b2a": float(a["benign_to_attack_rate"]), "seen_a2b": float(a["attack_to_benign_rate"]), "seen_attack_recall": float(a["binary_attack_recall"]),
           "seen_family_f1": float(a["family_macro_f1"]), "seen_fine_f1": float(a["fine_macro_f1"]), "seen_ece": float(a["ece15"]),
           "seen_maxprob": float(softmax_np(lg).max(axis=1).mean())}
    out.update({f"seen_recall_{f}": float((pf[S.fam_y == f] == f).mean()) for f in S.seen_families})
    lu = forward_logits(model, S.un_x); pu = lu.argmax(axis=1); p = softmax_np(lu); fu = S.fam[pu]
    out.update({"unseen_rows": int(len(pu)), "unseen_detect": float((pu != S.benign).mean()), "unseen_attack_mass": float((1 - p[:, S.benign]).mean()),
                "unseen_maxprob": float(p.max(axis=1).mean())})
    out.update({f"absorbed_{f}": float((fu == f).mean()) for f in S.seen_families})
    return out


def accepted(row):
    return bool(row["seen_b2a"] <= ACCEPT["b2a_max"] and row["seen_attack_recall"] >= ACCEPT["attack_recall_min"] and row["seen_fine_f1"] >= ACCEPT["fine_f1_min"])


def valid_order(table, column, min_w):
    t = table.sort_values(column, ascending=True); remaining = t.groupby("module_path")["group_id"].count().to_dict(); order = []
    for row in t.itertuples():
        layer = str(row.module_path)
        if remaining[layer] - 1 < min_w: continue
        remaining[layer] -= 1; order.append((layer, int(row.channel_index)))
    return order


def prune_map_of(pairs):
    pm = {}
    for layer, ch in pairs: pm.setdefault(str(layer), []).append(int(ch))
    return {l: sorted(c) for l, c in pm.items()}


def widths(st):
    return "/".join(str(m.out_channels) for _, m in st.named_modules() if isinstance(m, nn.Conv1d))


SETTINGS = {f: Setting(f) for f in FAMILIES}
for f, S in SETTINGS.items():
    print(f"held out {f:9s}: classes {len(S.unseen)} | remaining classes {S.K} | train rows {S.n_train:,} | eval rows seen {len(S.ex_y):,} unseen {len(S.un_y):,} | remaining families {S.seen_families}")

In [ ]:
# Stage 4 - the runs: per (architecture, family, seed) one teacher, one dense reference, two students (resumable; teachers cached)
RUNS = OUT / "N49_runs.csv"
rows = pd.read_csv(RUNS).to_dict("records") if RUNS.exists() else []
done = {(r["architecture"], r["family"], int(r["seed"]), r["model"]) for r in rows}
MODELS = ["teacher_raw", "dense"] + [f"{m}_b{int(round(b * 100))}" for m in METHODS for b in BUDGETS]
struct_path = OUT / "N49_structures.csv"; struct_rows = pd.read_csv(struct_path).to_dict("records") if struct_path.exists() else []
(OUT / "structures").mkdir(exist_ok=True)
RUN_FAMILIES = FAMILIES          # narrow this list to split the run across sessions, e.g. ["dos"]; rerun with the full list to finish


def save_row(row):
    rows.append(row); done.add((row["architecture"], row["family"], int(row["seed"]), row["model"])); pd.DataFrame(rows).to_csv(RUNS, index=False)
    print(f"  {row['model']:14s}: unseen detect {row['unseen_detect']:.3f} (attack mass {row['unseen_attack_mass']:.3f}) | seen HSR {row['seen_hsr_balanced']:.4f} "
          f"E {row['seen_b2a']:.4f} M {row['seen_a2b']:.4f} famF1 {row['seen_family_f1']:.3f} | {row['seconds']:.0f}s")


def teacher_for(arch, S, seed):
    ck = TEACHER_DIR / f"{arch}_{S.family}_s{seed}.pt"; inner = S.make(arch)
    if ck.exists():
        inner.load_state_dict(torch.load(ck, map_location="cpu", weights_only=False)["state_dict"]); return inner.to(DEVICE).eval(), True
    torch.manual_seed(seed); np.random.seed(seed); model = S.make(arch).to(DEVICE); train_epochs(model, S.loader(seed), T_EPOCHS, S.class_w[arch])
    torch.save({"state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, "architecture": arch, "family": S.family, "seed": seed, "classes": S.names}, ck)
    return model.eval(), False


for family, arch, seed in itertools.product(RUN_FAMILIES, ARCHS, SEEDS):
    if all((arch, family, seed, m) in done for m in MODELS): continue
    S = SETTINGS[family]; base = {"architecture": arch, "family": family, "seed": seed}
    print(f"\n== {arch} | held out {family} | seed {seed}")
    t0 = time.time(); teacher, cached = teacher_for(arch, S, seed)
    if (arch, family, seed, "teacher_raw") not in done:
        tr = copy.deepcopy(teacher); recalibrate_bn(tr, S.cal_loader); a = audit(tr, S)
        save_row({**base, "model": "teacher_raw", "method": "dense", "budget": 0.0, "parameters": int(count_parameters(teacher)), "realised_reduction": 0.0,
                  "teacher_accepted": accepted(a), "teacher_cached": cached, "seconds": round(time.time() - t0, 1), **a}); del tr
    teacher_row = next(r for r in rows if (r["architecture"], r["family"], int(r["seed"]), r["model"]) == (arch, family, seed, "teacher_raw"))
    acc = bool(teacher_row["teacher_accepted"])
    if not acc: print("  teacher FAILS the acceptance rule; its cell is excluded from the verdict and listed")
    if (arch, family, seed, "dense") not in done:
        t0 = time.time(); torch.manual_seed(seed + 1); dense = copy.deepcopy(teacher); train_epochs(dense, S.loader(seed + 1), R_EPOCHS, S.class_w[arch]); recalibrate_bn(dense, S.cal_loader)
        save_row({**base, "model": "dense", "method": "dense", "budget": 0.0, "parameters": int(count_parameters(dense)), "realised_reduction": 0.0,
                  "teacher_accepted": acc, "teacher_cached": cached, "seconds": round(time.time() - t0, 1), **audit(dense, S)}); del dense
    # scores on the teacher (remaining-class training batches), removal orders, calibrated structures
    groups = enumerate_cnn1d_channel_groups(teacher, S.example)
    mag = magnitude_scores(teacher, groups)
    grad = gradient_saliency_scores(teacher, S.loader(seed + 2), groups, device=DEVICE, max_batches=FISHER_BATCHES, criterion=nn.CrossEntropyLoss(weight=S.class_w[arch]))
    for t in (groups, mag, grad): t["group_id"] = t["group_id"].astype(str)
    table = groups[["group_id", "module_path", "channel_index"]].merge(mag, on="group_id", validate="1:1").merge(grad[["group_id", "fisher"]], on="group_id", validate="1:1")
    m0 = profile_forward_flops(teacher, S.example)["flops_per_item"]
    for method in METHODS:
        order = valid_order(table, method, MIN_W)
        build = lambda k, o=order: prune_cnn1d_channels(teacher, prune_map_of(o[:k]), S.example, minimum_remaining_per_layer=MIN_W)[0].to(DEVICE)
        realised = lambda st: 1.0 - profile_forward_flops(st, S.example)["flops_per_item"] / m0
        for budget in BUDGETS:
            name = f"{method}_b{int(round(budget * 100))}"
            if (arch, family, seed, name) in done: continue
            t0 = time.time(); cal = calibrate_prefix_count(order, target_reduction=budget, build_model=build, realised_reduction=realised, tolerance=TOL)
            assert cal.within_tolerance, f"{arch}/{family}/s{seed}/{name}: realised {cal.realised_reduction:.4f} is outside the tolerance"
            torch.manual_seed(seed + 1); st = build(cal.prefix_length)
            struct_rows = [r for r in struct_rows if (r["architecture"], r["family"], int(r["seed"]), r["model"]) != (arch, family, seed, name)]
            struct_rows.append({**base, "model": name, "k": cal.prefix_length, "realised_reduction": cal.realised_reduction, "widths": widths(st), "parameters": int(count_parameters(st))})
            pd.DataFrame(struct_rows).to_csv(struct_path, index=False)
            pd.DataFrame([{"module_path": l, "channel_index": c} for l, c in order[:cal.prefix_length]]).to_csv(OUT / f"structures/N49_{arch}_{family}_s{seed}_{name}_removed_groups.csv", index=False)
            train_epochs(st, S.loader(seed + 1), R_EPOCHS, S.class_w[arch]); recalibrate_bn(st, S.cal_loader)
            save_row({**base, "model": name, "method": method, "budget": budget, "parameters": int(count_parameters(st)), "realised_reduction": cal.realised_reduction,
                      "teacher_accepted": acc, "teacher_cached": cached, "seconds": round(time.time() - t0, 1), **audit(st, S)}); del st
    del teacher; gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None
print("\nrows:", len(rows), "of", len(ARCHS) * len(FAMILIES) * len(SEEDS) * len(MODELS))

In [ ]:
# Stage 5 - pooled analysis at nine seeds and verdict (rules exactly as pre-registered in stage 2)
runs_new = pd.read_csv(RUNS)
EXPECTED = len(ARCHS) * len(FAMILIES) * len(SEEDS) * len(MODELS)
assert len(runs_new) == EXPECTED, f"expected {EXPECTED} rows, found {len(runs_new)}: finish stage 4 first"
old = pd.read_csv(IN47 / "N47_runs.csv"); old = old[old.family == "recon"]
assert sorted(old.seed.unique()) == [101, 211, 307] and len(old) == 24, "notebook 47 reconnaissance rows not as expected"
common_cols = [c for c in runs_new.columns if c in old.columns]
runs = pd.concat([old[common_cols], runs_new[common_cols]], ignore_index=True); runs["teacher_accepted"] = runs["teacher_accepted"].astype(bool)
runs.to_csv(OUT / "N49_pooled_runs.csv", index=False)
excluded = runs[(runs.model == "teacher_raw") & (~runs.teacher_accepted)][["architecture", "family", "seed", "seen_b2a", "seen_attack_recall", "seen_fine_f1"]]
ok = runs[runs.teacher_accepted]


def paired(arch, model, col, seeds=None):
    q = ok if seeds is None else ok[ok.seed.isin(seeds)]
    s = q[(q.architecture == arch) & (q.family == "recon") & (q.model == model)].set_index("seed")[col]
    d = q[(q.architecture == arch) & (q.family == "recon") & (q.model == "dense")].set_index("seed")[col]
    common = s.index.intersection(d.index); g = (d.loc[common] - s.loc[common]).to_numpy(); n = len(g)
    hw = stats.t.ppf(0.975, n - 1) * g.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan
    return {"n": int(n), "mean": float(g.mean()), "lo": float(g.mean() - hw), "hi": float(g.mean() + hw), "n_student_lower": int((g > 0).sum())}


cells_, H = [], {}
for arch in ARCHS:
    reads = {}
    for method in METHODS:
        model = f"{method}_b90"; g = paired(arch, model, "unseen_detect"); g3 = paired(arch, model, "unseen_detect", [101, 211, 307]); h = paired(arch, model, "seen_hsr_balanced")
        e = paired(arch, model, "seen_b2a")
        d = ok[(ok.architecture == arch) & (ok.family == "recon") & (ok.model == "dense")]; s = ok[(ok.architecture == arch) & (ok.family == "recon") & (ok.model == model)]
        matched = bool(abs(h["mean"]) <= MATCH)
        read = "less" if (g["mean"] >= GAP and g["lo"] > 0 and matched) else ("more" if g["hi"] < 0 else "level")
        reads[method] = read
        cells_.append({"architecture": arch, "method": method, "n_seeds": g["n"], "dense_unseen_detect": float(d.unseen_detect.mean()), "student_unseen_detect": float(s.unseen_detect.mean()),
                       "gap": g["mean"], "gap_lo": g["lo"], "gap_hi": g["hi"], "seeds_student_lower": g["n_student_lower"], "gap3": g3["mean"], "gap3_lo": g3["lo"], "gap3_hi": g3["hi"],
                       "matched": matched, "hsr_gap_student_minus_dense": -h["mean"], "seen_E_diff_student_minus_dense": -e["mean"], "reading": read,
                       "dense_to_benign": float(d.absorbed_benign.mean()), "student_to_benign": float(s.absorbed_benign.mean()),
                       "dense_attack_mass": float(d.unseen_attack_mass.mean()), "student_attack_mass": float(s.unseen_attack_mass.mean())})
    per = "pruned_raise_less" if all(r == "less" for r in reads.values()) else ("pruned_raise_more" if all(r == "more" for r in reads.values()) else "mixed")
    H[arch] = {"reading": per, "by_rule": reads}
CELLS = pd.DataFrame(cells_); CELLS.to_csv(OUT / "N49_cells.csv", index=False)
Z = {"Z1_fisher_less_on_some_network": bool(any(H[a]["by_rule"]["fisher"] == "less" for a in ARCHS))}
verdict = {"notebook": "49_iomt_unseen_recon_seeds", "by_architecture": H, "expectation": Z, "cells": CELLS.round(4).to_dict("records"),
           "excluded_cells": excluded.round(4).to_dict("records"), "teachers_accepted": int(runs_new[runs_new.model == "teacher_raw"].teacher_accepted.astype(bool).sum()),
           "teachers": int((runs_new.model == "teacher_raw").sum()), "prereg": PREREG}
(OUT / "N49_verdict.json").write_text(json.dumps(verdict, indent=2, default=float))
print("teachers accepted (new):", verdict["teachers_accepted"], "of", verdict["teachers"], "| excluded cells (pooled):", len(excluded))
for arch in ARCHS: print(f"{arch}: {H[arch]['reading']} {H[arch]['by_rule']}")
print("expectation:", Z)
print(CELLS[["architecture", "method", "n_seeds", "dense_unseen_detect", "student_unseen_detect", "gap", "gap_lo", "gap_hi", "seeds_student_lower", "gap3", "matched", "seen_E_diff_student_minus_dense", "reading"]].round(4).to_string(index=False))

In [ ]:
# Stage 6 - figure (600 dpi, paper names): unseen reconnaissance raised as attack, per seed, dense vs students
plt.rcParams.update({"font.family": "serif", "font.serif": ["Liberation Serif", "DejaVu Serif"], "font.size": 7, "axes.labelsize": 7,
                     "xtick.labelsize": 6.5, "ytick.labelsize": 6.5, "legend.fontsize": 6, "pdf.fonttype": 42, "axes.linewidth": 0.6})
COLR = {"magnitude_b90": "#E69F00", "fisher_b90": "#009E73"}; LAB = {"magnitude_b90": "magnitude 90%", "fisher_b90": "Fisher 90%"}
fig, axes = plt.subplots(1, 2, figsize=(3.5, 1.75), sharey=True, gridspec_kw={"wspace": 0.18})
for ax, arch in zip(axes, ARCHS):
    d = ok[(ok.architecture == arch) & (ok.family == "recon") & (ok.model == "dense")].set_index("seed").unseen_detect * 100
    ax.plot([0, 100], [0, 100], color="#B8B8B8", lw=0.6, zorder=1)
    for mod in ["magnitude_b90", "fisher_b90"]:
        s = ok[(ok.architecture == arch) & (ok.family == "recon") & (ok.model == mod)].set_index("seed").unseen_detect * 100
        c = s.index.intersection(d.index); ax.scatter(d.loc[c], s.loc[c], s=9, color=COLR[mod], label=LAB[mod], zorder=3)
    ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.set_xticks([0, 25, 50, 75, 100]); ax.set_title(f"{arch} network", fontsize=7, pad=2); ax.set_xlabel("dense raises (%)")
    ax.spines[["top", "right"]].set_visible(False); ax.set_aspect("equal")
axes[0].set_ylabel("pruned raises (%)"); axes[0].legend(frameon=False, loc="upper left", handlelength=1.0)
fig.savefig(OUT / "N49_recon.png", dpi=600, bbox_inches="tight"); fig.savefig(OUT / "N49_recon.pdf", bbox_inches="tight"); plt.show()
print("written ->", OUT)

In [ ]:
# Stage 7 - commit the results (standard commit cell; the message is filled in from the verdict)
v = json.load(open(OUT / "N49_verdict.json"))
pre = subprocess.run(["git", "log", "-1", "--format=%h", "--", "results/saber/49_iomt_unseen_recon_seeds/N49_PREREGISTRATION.json"], capture_output=True, text=True, cwd=REPO).stdout.strip() or "see log"
txt = "; ".join(f"{c['architecture']}/{NAME[c['method']]}: dense {100 * c['dense_unseen_detect']:.1f}% student {100 * c['student_unseen_detect']:.1f}% "
                f"(gap {100 * c['gap']:+.1f} [{100 * c['gap_lo']:+.1f}, {100 * c['gap_hi']:+.1f}] pts, {c['n_seeds']} seeds, student lower on {c['seeds_student_lower']}, matched {c['matched']}, reads {c['reading']})" for c in v["cells"])
msg = (f"NB49 results (pre-registration {pre}): unseen reconnaissance on CIC-IoMT-2024 at nine seeds (six new; teachers accepted {v['teachers_accepted']} of {v['teachers']}, "
       f"excluded cells {len(v['excluded_cells'])}). shallow {v['by_architecture']['shallow']['reading']}, deep {v['by_architecture']['deep']['reading']}. {txt}. "
       f"Expectation Z1 (Fisher reads less on some network) {'HELD' if v['expectation']['Z1_fisher_less_on_some_network'] else 'NOT HELD'}.")
print(msg)
os.chdir(REPO)
r = subprocess.run(["python", "tools/commit_cell.py", msg], capture_output=True, text=True)
print(r.stdout); print(r.stderr)